In [1]:
import batting_statlines as stat 
import pandas as pd 
import numpy as np 
import mysql.connector as sql
from IPython import display

pd.options.display.max_columns = None

In [2]:
fangraphs_board = pd.read_csv('pitching_1947_2001.csv')

## Here We Go!

The first thing that I want to do, is to group everybody into teams. In order to do this analysis, I'm going to be looking at Johnson and Schilling's numbers together as a single aggregate. I think that would be the best, becuase I'm going to compare him to other top 2 pitchers on teams so we're really comparing "groups" of players rather than individual players.

### The Plan

What I plan on doing is first, create the aggregate values for each column, for the top two pitchers in a team each year. After I do that I want to rank each pair of staff aces for each statistic. From there, create an average score for how they rank in each statistic to find the most "dominant" pairing of pitchers. Then, look across each year and see who has the highest "dominance" score.

In [3]:
grouped_by_teams = stat.order_by(fangraphs_board, ['team', 'season'], True)
grouped_by_teams = grouped_by_teams[grouped_by_teams.team != '- - -']
grouped_by_teams.rename(columns={'season': 'yearID', 'team': 'teamID'}, inplace = True)
grouped_by_teams

,yearID,name,teamID,W,L,G,GS,IP,WHIP,K/9,K%,BB/9,BB%,HR/9,HR/FB,BABIP,ERA,FIP,xFIP,ERA-,FIP-,xFIP-,K/9+,BB/9+,K/BB+,WHIP+,K%+,BB%+,WAR,playerid
289,1961,Ken McBride,Angels,12,15,38,36,241.2,1.37,6.70,17.3%,3.80,9.8%,1.04,NaN,0.279,3.65,3.94,NaN,84,90,NaN,129,103,125.0,99,130,104,4.1,1008344
290,1961,Eli Grba,Angels,11,13,40,30,211.2,1.47,4.46,11.2%,4.85,12.1%,1.11,NaN,0.248,4.25,4.89,NaN,98,111,NaN,86,132,65.0,106,83,128,1.4,1004979
291,1962,Dean Chance,Angels,14,10,50,24,206.2,1.26,5.53,14.7%,2.87,7.6%,0.61,NaN,0.278,2.96,3.30,NaN,79,85,NaN,105,82,128.0,93,107,84,3.5,1002130
292,1962,Bo Belinsky,Angels,10,11,33,31,187.1,1.45,6.97,17.3%,5.86,14.5%,0.58,NaN,0.250,3.56,4.06,NaN,94,105,NaN,132,167,79.0,106,126,160,1.8,1000791
293,1962,Eli Grba,Angels,8,9,40,29,176.1,1.47,4.59,11.5%,3.83,9.6%,0.97,NaN,0.278,4.54,4.31,NaN,121,112,NaN,87,109,80.0,108,84,105,1.1,1004979
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5409,2017,Luis Severino,Yankees,14,6,31,31,193.1,1.04,10.71,29.4%,2.37,6.5%,0.98,14.0%,0.272,2.98,3.07,3.04,68,68,69.0,129,74,175.0,78,136,78,5.6,15890
5410,2017,Masahiro Tanaka,Yankees,13,12,30,30,178.1,1.24,9.79,25.8%,2.07,5.5%,1.77,21.2%,0.305,4.74,4.34,3.44,109,97,78.0,118,64,183.0,93,120,65,2.6,15764
5411,2018,Luis Severino,Yankees,19,8,32,32,191.1,1.14,10.35,28.2%,2.16,5.9%,0.89,11.4%,0.314,3.39,2.95,3.10,80,68,73.0,122,68,180.0,87,128,71,5.4,15890
5412,2019,Masahiro Tanaka,Yankees,11,8,31,31,179.0,1.23,7.39,19.8%,1.96,5.2%,1.41,15.6%,0.290,4.47,4.29,4.25,97,92,93.0,84,60,141.0,91,87,62,3.2,15764


In [8]:
year2001 = grouped_by_teams[grouped_by_teams.yearID == 2001].reset_index(drop=True)
year2001.head()

,yearID,name,teamID,W,L,G,GS,IP,WHIP,K/9,K%,BB/9,BB%,HR/9,HR/FB,BABIP,ERA,FIP,xFIP,ERA-,FIP-,xFIP-,K/9+,BB/9+,K/BB+,WHIP+,K%+,BB%+,WAR,playerid
0,2001,Jarrod Washburn,Angels,11,10,30,30,193.1,1.29,5.87,15.5%,2.51,6.6%,1.16,NaN,0.285,3.77,4.37,NaN,85,98,NaN,91,78,117.0,93,94,80,2.5,40
1,2001,Ramon Ortiz,Angels,13,11,32,32,208.2,1.43,5.82,14.7%,3.28,8.3%,1.08,NaN,0.296,4.36,4.58,NaN,98,103,NaN,90,101,89.0,103,89,100,2.3,27
2,2001,Ismael Valdez,Angels,9,13,27,27,163.2,1.39,5.50,14.3%,2.75,7.2%,1.10,NaN,0.301,4.45,4.48,NaN,100,101,NaN,85,85,100.0,100,87,86,2.0,1283
3,2001,Pat Rapp,Angels,5,12,28,28,165.0,1.41,4.25,11.0%,3.76,9.7%,1.04,NaN,0.268,4.80,4.89,NaN,108,110,NaN,66,116,57.0,102,67,117,1.3,1010700
4,2001,Scott Schoeneweis,Angels,10,11,32,32,205.1,1.48,4.56,11.4%,3.37,8.5%,0.92,NaN,0.297,5.08,4.70,NaN,114,106,NaN,71,104,68.0,106,69,102,2.0,33


In [9]:
teams_w_mult = {}
for team in year2001.teamID.unique():
    counter = 0
    teams_df = year2001[year2001.teamID == team]
    for name in teams_df.name.unique():
        counter += 1
    teams_w_mult[team] = counter
print(teams_w_mult)

{'Angels': 5, 'Astros': 2, 'Athletics': 4, 'Blue Jays': 2, 'Braves': 3, 'Brewers': 2, 'Cardinals': 3, 'Cubs': 4, 'Devil Rays': 1, 'Diamondbacks': 2, 'Dodgers': 1, 'Expos': 2, 'Giants': 3, 'Indians': 2, 'Mariners': 3, 'Marlins': 4, 'Mets': 4, 'Orioles': 2, 'Padres': 2, 'Phillies': 2, 'Pirates': 2, 'Rangers': 2, 'Red Sox': 1, 'Reds': 2, 'Rockies': 2, 'Royals': 2, 'Tigers': 2, 'Twins': 3, 'White Sox': 1, 'Yankees': 3}


In [26]:
angels_2001 = year2001[year2001.teamID == 'Angels']
angels_2001 = angels_2001.head(2)
angels_2001

,yearID,name,teamID,W,L,G,GS,IP,WHIP,K/9,K%,BB/9,BB%,HR/9,HR/FB,BABIP,ERA,FIP,xFIP,ERA-,FIP-,xFIP-,K/9+,BB/9+,K/BB+,WHIP+,K%+,BB%+,WAR,playerid
0,2001,Jarrod Washburn,Angels,11,10,30,30,193.1,1.29,5.87,15.5%,2.51,6.6%,1.16,NaN,0.285,3.77,4.37,NaN,85,98,NaN,91,78,117.0,93,94,80,2.5,40
1,2001,Ramon Ortiz,Angels,13,11,32,32,208.2,1.43,5.82,14.7%,3.28,8.3%,1.08,NaN,0.296,4.36,4.58,NaN,98,103,NaN,90,101,89.0,103,89,100,2.3,27


In [29]:
angels_2001.WAR.avg()

AttributeError: 'Series' object has no attribute 'avg'